In [1]:
import textwrap
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Legend text labels wrapped to roughly half the original visual width
labels = [
    "perfect calibration",
    textwrap.fill(
        "LOESS on pseudo-observations, median of the out-of-bag replicates (2.5-97.5 band)",
        width=34,
    ),
    textwrap.fill(
        "risk quartiles of the pooled out-of-bag risk, 1 - KM (95% CI)",
        width=34,
    ),
]

# Replicate the styling of the legend handles
handles = [
    Line2D(
        [0],
        [0],
        color="#627882",
        linestyle=(0, (4, 3)),
        linewidth=1.5,
    ),
    Line2D(
        [0],
        [0],
        color="#0072B2",
        linestyle="-",
        linewidth=2.0,
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="white",
        markeredgecolor="#D55E00",
        markeredgewidth=1.6,
        markersize=5,
    ),
]

# Render only the legend
fig = plt.figure(figsize=(2.6, 1.5))
legend = fig.legend(
    handles=handles,
    labels=labels,
    loc="center",
    frameon=False,
    fontsize=7.5,
    handlelength=1.6,
    handletextpad=0.5,
    labelspacing=0.5,
    borderaxespad=0,
    borderpad=0,
)

# Export tightly cropped to PDF
fig.savefig(
    "legend_half_width.pdf",
    format="pdf",
    bbox_inches=legend.get_window_extent().transformed(
        fig.dpi_scale_trans.inverted()
    ),
    pad_inches=0.02,
)
plt.close(fig)

# Join figE with the half-width legend

Takes the already-rendered `calibration/figE_calibration_5y.pdf` (produced by
`calibration.ipynb`), drops its baked-in bottom legend, and places the compact legend
from the cell above on its left, vertically centred against the plot. Composition is
done page-to-page with PyMuPDF so both halves stay vector - nothing is rasterised.

In [2]:
import os
import fitz  # PyMuPDF

FIGE_SRC = os.path.join("calibration", "figE_calibration_5y.pdf")
LEGEND_SRC = "legend_half_width.pdf"
OUT_STEM = os.path.join("calibration", "figE_calibration_5y_joined")
GAP = 4  # pt between the legend and the plot
LEFT_PAD = 2  # pt of breathing room kept before the y-axis label


def _axes_rect(page):
    """The plot's spine rectangle - the largest stroked rect on the page."""
    rects = [d["rect"] for d in page.get_drawings() if "s" in d.get("type", "")]
    rects = [r for r in rects if r.width < page.rect.width or r.height < page.rect.height]
    return max(rects, key=lambda r: r.width * r.height)


def _crop_figure(page):
    """Crop out the bottom-of-figure legend and the dead left margin.

    Keeps the x-axis label above the (now-removed) legend, and the y-axis
    label / tick labels on the left, but trims the empty canvas padding
    matplotlib leaves before them.
    """
    axes = _axes_rect(page)
    words = page.get_text("words")
    xlabel_bottom = max(
        w[3] for w in words
        if w[1] > axes.y1 and w[3] < axes.y1 + 40
    )
    bottom = xlabel_bottom + 4
    plot_words = [w for w in words if w[3] <= axes.y1 + 2]
    left = min(w[0] for w in plot_words) - LEFT_PAD
    return fitz.Rect(left, page.rect.y0, page.rect.x1, bottom)


src_e = fitz.open(FIGE_SRC)
page_e = src_e[0]
crop_e = _crop_figure(page_e)
page_e.set_cropbox(crop_e)
w_e, h_e = page_e.rect.width, page_e.rect.height

src_l = fitz.open(LEGEND_SRC)
page_l = src_l[0]
w_l, h_l = page_l.rect.width, page_l.rect.height

W, H = w_l + GAP + w_e, max(h_e, h_l)
out = fitz.open()
page = out.new_page(width=W, height=H)

legend_rect = fitz.Rect(0, (H - h_l) / 2, w_l, (H - h_l) / 2 + h_l)
fige_rect = fitz.Rect(w_l + GAP, (H - h_e) / 2, w_l + GAP + w_e, (H - h_e) / 2 + h_e)

page.show_pdf_page(legend_rect, src_l, 0)
page.show_pdf_page(fige_rect, src_e, 0)

out.save(f"{OUT_STEM}.pdf")
pix = page.get_pixmap(matrix=fitz.Matrix(300 / 72, 300 / 72))
pix.save(f"{OUT_STEM}.png")
print(f"wrote {OUT_STEM}.pdf / .png  ({W:.0f}x{H:.0f} pt)")

wrote calibration\figE_calibration_5y_joined.pdf / .png  (465x257 pt)
